In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("catalogo", "catalog_tr")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("tabla_silver", "transactions_silver")
dbutils.widgets.text("esquema_gold", "golden")

catalogo = dbutils.widgets.get("catalogo")
tabla_silver = f"{catalogo}.{dbutils.widgets.get('esquema_silver')}.{dbutils.widgets.get('tabla_silver')}"
esquema_gold = f"{catalogo}.{dbutils.widgets.get('esquema_gold')}"
print("Source:", tabla_silver)
print("Target schema:", esquema_gold)

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table(tabla_silver)
print("Silver rows:", df_silver.count())

gold_tables = {}

def save_gold(df, name):
    """Overwrite a gold Delta table and register it for validation."""
    full_name = f"{esquema_gold}.{name}"
    (df.withColumn("gold_load_ts", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(full_name))
    gold_tables[name] = full_name
    print("Saved", full_name)

# Common KPI expressions
def kpis():
    return [
        F.countDistinct("transaction_id").alias("transactions"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.sum("total_amount") / F.countDistinct("transaction_id"), 2).alias("avg_ticket"),
        F.countDistinct("customer_id").alias("unique_customers"),
    ]

In [0]:
# Clean transaction-level fact for drill-down and flexible dashboard filters
fact_sales = df_silver.select(
    "transaction_id", "transaction_date", "transaction_day", "transaction_hour", "day_of_week",
    F.date_trunc("week", "transaction_date").cast("date").alias("transaction_week"),
    "customer_id", "product_id", "product_name", "category",
    "quantity", "unit_price", "total_amount", "store_location", "payment_method",
)
save_gold(fact_sales, "fact_sales")

In [0]:
# Trend table: one row per day x store x category (supports trend lines with filters)
sales_daily = (
    df_silver.groupBy("transaction_day", "day_of_week", "store_location", "category")
    .agg(*kpis())
)
save_gold(sales_daily, "sales_daily")

In [0]:
from pyspark.sql.window import Window

total_revenue = df_silver.agg(F.sum("total_amount")).first()[0]

product_performance = (
    df_silver.groupBy("product_id", "product_name", "category")
    .agg(*kpis(), F.round(F.avg("unit_price"), 2).alias("avg_unit_price"))
    .withColumn("revenue_share_pct", F.round(F.col("revenue") / F.lit(total_revenue) * 100, 2))
    .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(F.desc("revenue"))))
)
save_gold(product_performance, "product_performance")

In [0]:
# Top category per store
w_cat = Window.partitionBy("store_location").orderBy(F.desc("cat_revenue"), F.asc("category"))
top_category = (
    df_silver.groupBy("store_location", "category").agg(F.sum("total_amount").alias("cat_revenue"))
    .withColumn("rn", F.row_number().over(w_cat)).filter("rn = 1")
    .select("store_location", F.col("category").alias("top_category"))
)

store_performance = (
    df_silver.groupBy("store_location").agg(*kpis())
    .join(top_category, "store_location", "left")
    .withColumn("revenue_share_pct", F.round(F.col("revenue") / F.lit(total_revenue) * 100, 2))
    .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(F.desc("revenue"))))
)
save_gold(store_performance, "store_performance")

In [0]:
payment_by_store = df_silver.groupBy("store_location", "payment_method").agg(*kpis())
save_gold(payment_by_store, "payment_method_by_store")

In [0]:
# Heatmap table: weekday x hour (day_of_week_num 1=Monday ... 7=Sunday for sorting)
sales_hourly = (
    df_silver
    .withColumn("day_of_week_num", F.weekday("transaction_date") + 1)
    .groupBy("day_of_week_num", "day_of_week", "transaction_hour")
    .agg(*kpis())
)
save_gold(sales_hourly, "sales_by_weekday_hour")

In [0]:
w_store = Window.partitionBy("customer_id").orderBy(F.desc("n"), F.asc("store_location"))
fav_store = (
    df_silver.groupBy("customer_id", "store_location").agg(F.count("*").alias("n"))
    .withColumn("rn", F.row_number().over(w_store)).filter("rn = 1")
    .select("customer_id", F.col("store_location").alias("favorite_store"))
)

customer_summary = (
    df_silver.groupBy("customer_id")
    .agg(
        F.countDistinct("transaction_id").alias("transactions"),
        F.sum("quantity").alias("units_bought"),
        F.round(F.sum("total_amount"), 2).alias("total_spent"),
        F.round(F.avg("total_amount"), 2).alias("avg_ticket"),
        F.countDistinct("category").alias("categories_bought"),
        F.min("transaction_date").alias("first_purchase"),
        F.max("transaction_date").alias("last_purchase"),
    )
    .join(fav_store, "customer_id", "left")
    .withColumn(
        "customer_segment",
        F.when(F.col("transactions") >= 3, "Frequent")
         .when(F.col("transactions") == 2, "Returning")
         .otherwise("One-time"),
    )
)
save_gold(customer_summary, "customer_summary")

In [0]:
# Revenue must reconcile with silver in every aggregated table
silver_revenue = round(total_revenue, 2)
revenue_col = {"customer_summary": "total_spent"}

rows = []
for name, full_name in gold_tables.items():
    t = spark.table(full_name)
    col = revenue_col.get(name, "total_amount" if name == "fact_sales" else "revenue")
    rev = t.agg(F.round(F.sum(col), 2)).first()[0]
    rows.append((full_name, t.count(), rev, silver_revenue, abs(rev - silver_revenue) < 0.05))

display(spark.createDataFrame(rows, "table string, rows long, revenue double, silver_revenue double, reconciles boolean"))